In [1]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

Looking in links: /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arc_agi-0.9.6-py3-none-any.whl
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arcengine-0.9.3-py3-none-any.whl (from arc-agi)
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/pillow-12.1.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from arc-agi)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatib

In [2]:
%%writefile /kaggle/working/my_agent.py

import hashlib
import logging
import os
import random
import time
import traceback
from collections import defaultdict, deque
from datetime import datetime
from typing import Any, Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from agents.agent import Agent
from arcengine import FrameData, GameAction, GameState


# ---------------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------------

def setup_experiment_directory(base_output_dir='runs'):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_dir = os.path.join(base_output_dir, timestamp)
    os.makedirs(base_dir, exist_ok=True)
    log_file = os.path.join(base_dir, 'logs.log')
    print(f"Experiment directory: {base_dir}")
    return base_dir, log_file


def get_environment_directory(base_dir, game_id):
    env_dir = os.path.join(base_dir, game_id)
    os.makedirs(env_dir, exist_ok=True)
    return env_dir


def setup_logging_for_experiment(log_file_path):
    root_logger = logging.getLogger()
    for handler in root_logger.handlers[:]:
        if isinstance(handler, logging.FileHandler):
            root_logger.removeHandler(handler)
            handler.close()
    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    fh = logging.FileHandler(log_file_path, mode="w")
    fh.setLevel(root_logger.level)
    fh.setFormatter(formatter)
    root_logger.addHandler(fh)


def frame_hash(frame_np: np.ndarray) -> str:
    """Fast hash of a boolean/int frame array."""
    return hashlib.md5(frame_np.tobytes()).hexdigest()


# ---------------------------------------------------------------------------
# Model: Residual CNN with lightweight cross-attention head
# ---------------------------------------------------------------------------

class ResidualBlock(nn.Module):
    """Standard pre-activation residual block."""
    def __init__(self, channels: int):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)

    def forward(self, x):
        residual = x
        out = self.conv1(F.relu(self.bn1(x)))
        out = self.conv2(F.relu(self.bn2(out)))
        return out + residual


class ActionModelV2(nn.Module):
    """
    Hybrid residual CNN for action/coordinate prediction.

    Input:  (B, C_in, 64, 64)  — one-hot colour planes + optional delta channel
    Output: (B, 5 + 64*64)     — logits for ACTION1-5 and 4096 coordinates

    Architecture rationale vs pure ViT:
      - Residual CNN gives translation-equivariant features cheaply.
      - A lightweight 1-head cross-attention on the flattened spatial tokens
        lets action head attend to the whole grid (global context) without
        the quadratic O(N²) cost of full ViT on the coordinate head.
      - Total params ~900K vs ~86M for ViT-B/16 — crucial for fast online training.
    """

    def __init__(self, input_channels: int = 18, grid_size: int = 64):
        super().__init__()
        self.grid_size = grid_size
        self.num_action_types = 5

        # --- Stem ---
        self.stem = nn.Sequential(
            nn.Conv2d(input_channels, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        # --- Residual backbone (64 → 128 → 256) ---
        self.stage1 = nn.Sequential(ResidualBlock(64), ResidualBlock(64))
        self.down1  = nn.Conv2d(64, 128, 2, stride=2, bias=False)   # 64→32

        self.stage2 = nn.Sequential(ResidualBlock(128), ResidualBlock(128))
        self.down2  = nn.Conv2d(128, 256, 2, stride=2, bias=False)  # 32→16

        self.stage3 = nn.Sequential(ResidualBlock(256), ResidualBlock(256))
        # spatial: 16×16, channels: 256  →  256 tokens of dim 256

        # --- Lightweight cross-attention (action queries → spatial tokens) ---
        # 5 learnable action queries attend to flattened 16×16 spatial tokens
        self.action_queries = nn.Parameter(torch.randn(1, 5, 256))
        self.attn = nn.MultiheadAttention(embed_dim=256, num_heads=4,
                                          batch_first=True, dropout=0.1)
        self.action_head = nn.Linear(256, 1)   # per-query logit

        # --- Coordinate head (fully convolutional, preserves 2D bias) ---
        # Upsample back to 64×64
        self.up1      = nn.ConvTranspose2d(256, 128, 2, stride=2)   # 16→32
        self.coord_r1 = ResidualBlock(128)
        self.up2      = nn.ConvTranspose2d(128, 64, 2, stride=2)    # 32→64
        self.coord_r2 = ResidualBlock(64)
        self.coord_out = nn.Conv2d(64, 1, 1)                        # → 64×64

        self.dropout = nn.Dropout(0.1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def forward(self, x):
        B = x.size(0)

        # Backbone
        h = self.stem(x)
        h = self.stage1(h)
        h = self.down1(h)
        h = self.stage2(h)
        h = self.down2(h)
        h = self.stage3(h)           # (B, 256, 16, 16)

        # --- Action head via cross-attention ---
        tokens = h.flatten(2).permute(0, 2, 1)           # (B, 256, 256)
        queries = self.action_queries.expand(B, -1, -1)  # (B, 5, 256)
        attn_out, _ = self.attn(queries, tokens, tokens) # (B, 5, 256)
        attn_out = self.dropout(attn_out)
        action_logits = self.action_head(attn_out).squeeze(-1)  # (B, 5)

        # --- Coordinate head (decoder) ---
        c = self.up1(h)
        c = self.coord_r1(c)
        c = self.up2(c)
        c = self.coord_r2(c)
        coord_logits = self.coord_out(c).view(B, -1)     # (B, 64*64)

        return torch.cat([action_logits, coord_logits], dim=1)  # (B, 5+4096)


# ---------------------------------------------------------------------------
# State Graph (from Rudakov et al., arXiv:2512.24156)
# ---------------------------------------------------------------------------

class StateGraph:
    """
    Directed graph of observed states and transitions.
    Nodes: frame hashes.
    Edges: (src_hash, action_idx) → dst_hash.

    Tracks:
      - Visited (state, action) pairs to avoid redundant exploration.
      - Visit counts per state for count-based intrinsic reward.
    """

    def __init__(self):
        self.edges: Dict[Tuple[str, int], str] = {}        # (src, act) → dst
        self.visit_count: Dict[str, int] = defaultdict(int)
        self.action_tried: Dict[str, set] = defaultdict(set)

    def record_transition(self, src_hash: str, action_idx: int, dst_hash: str):
        key = (src_hash, action_idx)
        self.edges[key] = dst_hash
        self.visit_count[dst_hash] += 1
        self.action_tried[src_hash].add(action_idx)

    def has_tried(self, state_hash: str, action_idx: int) -> bool:
        return action_idx in self.action_tried.get(state_hash, set())

    def novelty_bonus(self, dst_hash: str, base: float = 1.0) -> float:
        """Count-based intrinsic bonus: 1/sqrt(n) decays with visits."""
        n = self.visit_count.get(dst_hash, 0)
        return base / (np.sqrt(n + 1))

    def untried_action_mask(self, state_hash: str, num_actions: int) -> np.ndarray:
        """Returns 1 for untried actions, 0 for tried ones."""
        tried = self.action_tried.get(state_hash, set())
        mask = np.ones(num_actions, dtype=np.float32)
        for a in tried:
            if a < num_actions:
                mask[a] = 0.0
        return mask

    def reset(self):
        self.edges.clear()
        self.visit_count.clear()
        self.action_tried.clear()


# ---------------------------------------------------------------------------
# Prioritized Experience Replay
# ---------------------------------------------------------------------------

class PrioritizedReplayBuffer:
    """
    Simple priority replay: positive-reward transitions are oversampled.
    Priority = (reward + novelty_bonus + epsilon).
    No tree structure — small buffer makes linear sampling fine.
    """

    def __init__(self, maxlen: int = 200_000, alpha: float = 0.6):
        self.buffer: deque = deque(maxlen=maxlen)
        self.priorities: deque = deque(maxlen=maxlen)
        self.hashes: set = set()
        self.alpha = alpha
        self.eps = 1e-4

    def add(self, exp: dict, priority: float = 1.0):
        key = exp.get('hash', id(exp))
        if key in self.hashes:
            return
        self.hashes.add(key)
        self.buffer.append(exp)
        self.priorities.append(max(priority, self.eps) ** self.alpha)

    def sample(self, batch_size: int):
        if len(self.buffer) == 0:
            return []
        probs = np.array(self.priorities, dtype=np.float64)
        probs /= probs.sum()
        idxs = np.random.choice(len(self.buffer), size=min(batch_size, len(self.buffer)),
                                replace=False, p=probs)
        return [self.buffer[i] for i in idxs]

    def __len__(self):
        return len(self.buffer)

    def clear(self):
        self.buffer.clear()
        self.priorities.clear()
        self.hashes.clear()


# ---------------------------------------------------------------------------
# Agent
# ---------------------------------------------------------------------------

class MyAgent(Agent):
    """
    StochasticGoose v2: Residual CNN + state graph + count-based exploration.

    Improvements over v1:
    ─────────────────────
    1. ResNet backbone with cross-attention action head (better features,
       stable gradients, mild global context without ViT's data cost).
    2. State graph prunes already-tried (state, action) pairs → less wasted budget.
    3. Count-based intrinsic reward encourages visiting novel states.
    4. Frame delta channel: XOR of current vs previous frame fed as extra input
       makes it easy for the network to see *what changed*.
    5. Prioritized replay: positive-reward samples replayed more often.
    6. Level transfer: on level-up, freeze backbone for 50 steps then unfreeze.
       Preserves general visual features across levels.
    7. UCB-style bonus in action sampling (biases toward untried actions).
    """

    MAX_ACTIONS = float('inf')
    _MAX_FRAMES = 10

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        seed = int(time.time() * 1_000_000) + hash(self.game_id) % 1_000_000
        random.seed(seed)
        np.random.seed(seed % (2**32 - 1))
        torch.manual_seed(seed % (2**32 - 1))
        self.start_time = time.time()

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"[v2] Device: {self.device}")

        self.base_dir, log_file = setup_experiment_directory()
        setup_logging_for_experiment(log_file)
        env_dir = get_environment_directory(self.base_dir, self.game_id)
        self.logger = logging.getLogger(f"Agent_v2_{self.game_id}")

        self.save_action_visualizations = False

        # Grid / colour config
        self.grid_size    = 64
        self.num_colours  = 16
        self.num_coords   = self.grid_size * self.grid_size
        # +2 input channels: delta frame + visit-count heatmap
        self.input_channels = self.num_colours + 2

        # Model & optimiser
        self.action_model: Optional[nn.Module] = None
        self.optimizer    = None
        self._init_model()

        # Experience replay
        self.replay = PrioritizedReplayBuffer(maxlen=200_000)
        self.batch_size      = 64
        self.train_frequency = 5
        self.warmup_steps    = self.batch_size  # don't train until buffer full enough

        # State graph
        self.state_graph = StateGraph()

        # Tracking
        self.current_score   = -1
        self.prev_frame_np   = None   # uint8 (num_colours, H, W)
        self.prev_action_idx = None
        self.prev_state_hash = None
        self.level_start_action = 0
        self.backbone_frozen = False

        # Action list
        self.action_list = [
            GameAction.ACTION1, GameAction.ACTION2, GameAction.ACTION3,
            GameAction.ACTION4, GameAction.ACTION5,
        ]

        self.log_dir = env_dir
        self.logger.info(f"Agent v2 initialized for game_id={self.game_id}")

    # ------------------------------------------------------------------
    # Model management
    # ------------------------------------------------------------------

    def _init_model(self, transfer_weights: Optional[dict] = None):
        """Create (or reset) model. Optionally warm-start from previous weights."""
        self.action_model = ActionModelV2(
            input_channels=self.input_channels,
            grid_size=self.grid_size,
        ).to(self.device)

        if transfer_weights is not None:
            # Transfer backbone weights; re-initialise heads
            try:
                incompatible = self.action_model.load_state_dict(transfer_weights, strict=False)
                self.logger.info(f"Weight transfer: {incompatible}")
            except Exception as e:
                self.logger.warning(f"Transfer failed: {e}")

        self.optimizer = optim.AdamW(
            self.action_model.parameters(), lr=3e-4, weight_decay=1e-4
        )
        self.scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.optimizer, T_0=500, eta_min=1e-5
        )

    def _freeze_backbone(self, freeze: bool):
        for name, param in self.action_model.named_parameters():
            if any(n in name for n in ('stem', 'stage', 'down')):
                param.requires_grad = not freeze
        self.backbone_frozen = freeze

    # ------------------------------------------------------------------
    # Frame utilities
    # ------------------------------------------------------------------

    def append_frame(self, frame: FrameData) -> None:
        self.frames.append(frame)
        if len(self.frames) > self._MAX_FRAMES:
            self.frames = self.frames[-self._MAX_FRAMES:]
        if frame.guid:
            self.guid = frame.guid
        if hasattr(self, "recorder") and not self.is_playback:
            import json
            self.recorder.record(json.loads(frame.model_dump_json()))

    def _get_score(self, frame) -> int:
        return getattr(frame, 'score', None) or frame.levels_completed

    def _frame_to_numpy(self, frame_data: FrameData) -> Optional[np.ndarray]:
        """Convert frame to uint8 array of shape (num_colours, H, W)."""
        frame = np.array(frame_data.frame, dtype=np.int64)
        frame = frame[-1]  # latest timestep
        if frame.shape != (self.grid_size, self.grid_size):
            return None
        one_hot = np.zeros((self.num_colours, self.grid_size, self.grid_size), dtype=np.uint8)
        for c in range(self.num_colours):
            one_hot[c] = (frame == c).astype(np.uint8)
        return one_hot

    def _build_input_tensor(self, frame_np: np.ndarray, prev_np: Optional[np.ndarray],
                            state_hash: str) -> torch.Tensor:
        """
        Build (input_channels, H, W) input tensor:
          channels 0..15  : one-hot colour
          channel  16     : frame delta (XOR) vs previous frame, normalised
          channel  17     : log-normalised visit count heatmap (novelty signal)
        """
        float_frame = frame_np.astype(np.float32)  # (16, H, W)

        # Delta channel
        if prev_np is not None:
            delta = (frame_np != prev_np).any(axis=0).astype(np.float32)  # (H, W)
        else:
            delta = np.zeros((self.grid_size, self.grid_size), dtype=np.float32)
        delta = delta[np.newaxis]  # (1, H, W)

        # Visit-count heatmap (global: how many times each cell was "active")
        visit_count = self.state_graph.visit_count.get(state_hash, 0)
        visit_map = np.full((1, self.grid_size, self.grid_size),
                            np.log1p(visit_count) / 10.0, dtype=np.float32)

        inp = np.concatenate([float_frame, delta, visit_map], axis=0)  # (18, H, W)
        return torch.from_numpy(inp).to(self.device)

    # ------------------------------------------------------------------
    # Sampling
    # ------------------------------------------------------------------

    def _sample_action(self, combined_logits: torch.Tensor,
                       state_hash: str,
                       available_actions) -> Tuple:
        """
        Sample from (5 action + 4096 coord) logits with:
        - Availability masking
        - UCB-style bonus for untried (state, action) pairs
        - Count-based novelty weighting
        """
        action_logits = combined_logits[:5].clone()
        coord_logits  = combined_logits[5:].clone()

        # --- Availability mask ---
        action6_available = False
        if available_actions is not None and len(available_actions) > 0:
            action_mask = torch.full_like(action_logits, float('-inf'))
            for action in available_actions:
                aid = action.value if hasattr(action, 'value') else int(action)
                if 1 <= aid <= 5:
                    action_mask[aid - 1] = 0.0
                elif aid == 6:
                    action6_available = True
            action_logits = action_logits + action_mask
            if not action6_available:
                coord_logits = coord_logits + torch.full_like(coord_logits, float('-inf'))

        # --- UCB bonus: boost untried discrete actions ---
        untried_mask = self.state_graph.untried_action_mask(state_hash, 5)
        ucb_bonus = torch.from_numpy(untried_mask).to(self.device) * 2.0
        action_logits = action_logits + ucb_bonus  # encourage exploration

        # --- Convert to probabilities ---
        action_probs = torch.sigmoid(action_logits)
        coord_probs  = torch.sigmoid(coord_logits) / self.num_coords

        all_probs = torch.cat([action_probs, coord_probs])
        all_probs_sum = all_probs.sum()
        if all_probs_sum <= 0 or torch.isnan(all_probs_sum):
            # Fallback: uniform over discrete actions
            selected_idx = random.randint(0, 4)
            return selected_idx, None, None, action_probs.cpu().numpy()
        all_probs = all_probs / all_probs_sum

        all_probs_np = all_probs.cpu().numpy()
        selected_idx = np.random.choice(len(all_probs_np), p=all_probs_np)

        if selected_idx < 5:
            return selected_idx, None, None, all_probs_np
        else:
            coord_idx = selected_idx - 5
            y = coord_idx // self.grid_size
            x = coord_idx % self.grid_size
            return 5, (y, x), coord_idx, all_probs_np

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------

    def _train_step(self):
        if len(self.replay) < self.warmup_steps:
            return

        batch = self.replay.sample(self.batch_size)
        if not batch:
            return

        states  = torch.stack([
            torch.from_numpy(exp['state']).float().to(self.device) for exp in batch
        ])
        actions = torch.tensor(
            [exp['action_idx'] for exp in batch], dtype=torch.long, device=self.device
        )
        rewards = torch.tensor(
            [exp['reward'] for exp in batch], dtype=torch.float32, device=self.device
        )

        self.optimizer.zero_grad()
        combined_logits = self.action_model(states)
        selected_logits = combined_logits.gather(1, actions.unsqueeze(1)).squeeze(1)

        # Binary cross-entropy: predict whether this action caused a frame change
        main_loss = F.binary_cross_entropy_with_logits(selected_logits, rewards)

        # Entropy regularisation (prevent collapse)
        action_probs = torch.sigmoid(combined_logits[:, :5])
        coord_probs  = torch.sigmoid(combined_logits[:, 5:])
        ent_loss = -(0.001 * action_probs.mean() + 0.0001 * coord_probs.mean())

        loss = main_loss + ent_loss
        loss.backward()
        nn.utils.clip_grad_norm_(self.action_model.parameters(), max_norm=1.0)
        self.optimizer.step()
        self.scheduler.step()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ------------------------------------------------------------------
    # Agent interface
    # ------------------------------------------------------------------

    def _has_time_elapsed(self) -> bool:
        return (time.time() - self.start_time) >= 8 * 3600 - 5 * 60

    def is_done(self, frames, latest_frame) -> bool:
        try:
            return latest_frame.state is GameState.WIN or self._has_time_elapsed()
        except Exception as e:
            print(f"[is_done crash] {e}")
            return True

    def choose_action(self, frames, latest_frame):
        try:
            # --- Debug logging on first call ---
            if self.action_counter == 0:
                print(f"[v2 DEBUG] frame type: {type(latest_frame)}")
                print(f"[v2 DEBUG] state: {latest_frame.state}")
                print(f"[v2 DEBUG] levels_completed: {latest_frame.levels_completed}")
                avail = getattr(latest_frame, 'available_actions', 'N/A')
                print(f"[v2 DEBUG] available_actions: {avail}")

            # --- Level change detection ---
            current_level = self._get_score(latest_frame)
            if current_level != self.current_score:
                print(f"[v2] Level change: {self.current_score} → {current_level} "
                      f"(action {self.action_counter})")

                # Transfer backbone weights to new model
                old_weights = self.action_model.state_dict() if self.action_model else None
                self._init_model(transfer_weights=old_weights)

                # Freeze backbone for first 50 steps on new level
                self._freeze_backbone(True)
                self.level_start_action = self.action_counter

                # Clear experience buffer and state graph for new level
                self.replay.clear()
                self.state_graph.reset()
                self.prev_frame_np   = None
                self.prev_action_idx = None
                self.prev_state_hash = None
                self.current_score   = current_level

            # Unfreeze backbone after warmup on new level
            if self.backbone_frozen and (self.action_counter - self.level_start_action) >= 50:
                self._freeze_backbone(False)

            # --- Handle terminal/reset states ---
            if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
                self.prev_frame_np   = None
                self.prev_action_idx = None
                self.prev_state_hash = None
                act = GameAction.RESET
                act.reasoning = "Reset."
                return act

            # --- Convert frame ---
            frame_np = self._frame_to_numpy(latest_frame)
            if frame_np is None:
                act = random.choice(self.action_list)
                act.reasoning = "Bad frame shape, random fallback."
                return act

            state_hash = frame_hash(frame_np)

            # --- Record experience from previous action ---
            if self.prev_frame_np is not None and self.prev_action_idx is not None:
                # Extrinsic reward: did the frame change?
                frame_changed = not np.array_equal(self.prev_frame_np, frame_np)
                extrinsic     = 1.0 if frame_changed else 0.0

                # Intrinsic novelty bonus
                novelty = self.state_graph.novelty_bonus(state_hash)

                reward = extrinsic + novelty * 0.5  # combine, novelty weighted lower

                # Build input that was used for previous action
                inp_tensor = self._build_input_tensor(
                    self.prev_frame_np, None, self.prev_state_hash or ""
                )
                exp_hash = hashlib.md5(
                    self.prev_frame_np.tobytes() +
                    str(self.prev_action_idx).encode()
                ).hexdigest()

                self.replay.add({
                    'state':      inp_tensor.cpu().numpy(),
                    'action_idx': self.prev_action_idx,
                    'reward':     min(reward, 2.0),   # clip
                    'hash':       exp_hash,
                }, priority=reward + 1e-3)

                # Update state graph
                self.state_graph.record_transition(
                    self.prev_state_hash or "?",
                    self.prev_action_idx,
                    state_hash,
                )

            # --- Build input and predict ---
            input_tensor = self._build_input_tensor(frame_np, self.prev_frame_np, state_hash)

            available_actions = getattr(latest_frame, 'available_actions', None)

            with torch.no_grad():
                logits = self.action_model(input_tensor.unsqueeze(0)).squeeze(0)
                action_idx, coords, coord_idx, all_probs = self._sample_action(
                    logits, state_hash, available_actions
                )

            if action_idx < 5:
                selected = self.action_list[action_idx]
                selected.reasoning = (
                    f"{selected.name} p={all_probs[action_idx]:.3f} "
                    f"{'[untried]' if not self.state_graph.has_tried(state_hash, action_idx) else ''}"
                )
            else:
                selected = GameAction.ACTION6
                y, x = coords
                selected.set_data({"x": int(x), "y": int(y)})
                selected.reasoning = f"ACTION6 ({x},{y}) coord_idx={coord_idx}"

            # Save state
            self.prev_frame_np   = frame_np
            self.prev_action_idx = action_idx if action_idx < 5 else (5 + coord_idx)
            self.prev_state_hash = state_hash

            # --- Train ---
            if self.action_counter % self.train_frequency == 0:
                self._train_step()

            return selected

        except Exception as e:
            print(f"[v2 choose_action CRASH] action={self.action_counter}: {e}")
            traceback.print_exc()
            act = random.choice(self.action_list)
            act.reasoning = f"Error fallback: {e}"
            return act

Writing /kaggle/working/my_agent.py


In [3]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for gateway to be ready
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy repo to writable location
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Copy custom agent
    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Write a minimal __init__.py that only imports what we need
    # (the original eagerly imports templates with unmet deps like langgraph, smolagents)
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
""")

    # Write a .env file that overrides .env.example defaults
    # This is loaded second with override=True by main.py, so it wins
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    # Run agent
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent

In [4]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Non-competition mode: produce a dummy submission
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    submission.head()